In [ ]:
import os
from glob import glob
import geopandas as gpd
import pandas
import fiona
import numpy
import cartopy.crs as ccrs
import matplotlib.pyplot as plt
import rasterio
# from analysis_utils import *
from pathlib import Path
import pandas as pd
import re

import gc
import numpy as np
import geopandas as gpd

In [ ]:
base_path = Path("/Users/robynhaggis/Documents/Geospatial_analysis/dphil_papers")

In [ ]:
jamaica_metric_grid_crs = "EPSG:3448"

In [ ]:
# --- New mangrove layer stats in protected areas ---
mangroves_fon_path = base_path / "dphil_paper_3/inputs/forces_of_nature_mangroves/mangroves.shp"

mangroves_fon = gpd.read_file(mangroves_fon_path).to_crs(jamaica_metric_grid_crs)
mangroves_fon = mangroves_fon[mangroves_fon.geometry.notna()].copy()
mangroves_fon["geometry"] = mangroves_fon.geometry.make_valid()


In [ ]:
land_use = base_path / "dphil_common_cross_cutting/common_incoming_data/landcover/2013_landcover/2013_landuse_LandCover.shp"
terrestrial_landcover = gpd.read_file(land_use)
terrestrial_landcover = terrestrial_landcover.to_crs(jamaica_metric_grid_crs)
print(terrestrial_landcover.crs)

In [ ]:
terrestrial_landcover.plot()

In [ ]:
# Clean inputs
mang = mangroves_fon[["geometry"]].copy()
mang = mang[mang.geometry.notna()].copy()
mang["geometry"] = mang.geometry.make_valid()

land = terrestrial_landcover[["Classify", "geometry"]].copy()
land = land.dropna(subset=["Classify"])
land = land[land.geometry.notna()].copy()
land["geometry"] = land.geometry.make_valid()

# Dissolve mangroves to one footprint (avoids overlap double counting)
mang_union = mang.geometry.union_all()

summaries = {}

for distance in [100, 1000]:
    ring_geom = mang_union.buffer(distance).difference(mang_union)
    ring = gpd.GeoDataFrame(geometry=[ring_geom], crs=jamaica_metric_grid_crs)

    land_in_ring = gpd.clip(land, ring)
    land_in_ring["area_m2"] = land_in_ring.geometry.area

    summary = (
        land_in_ring.groupby("Classify", as_index=False)["area_m2"]
        .sum()
        .sort_values("area_m2", ascending=False)
    )

    total_ring_m2 = summary["area_m2"].sum()
    summary["area_km2"] = summary["area_m2"] / 1e6
    summary["% of surrounding ring"] = (summary["area_m2"] / total_ring_m2) * 100
    summary["distance_m"] = distance

    summaries[distance] = summary
    total_pct = summary["% of surrounding ring"].sum()
    print(f"{distance} m: total percentage = {total_pct:.10f}%")

    display(summary)

    out_csv = base_path / f"dphil_paper_3/processed_data/mangrove_surrounding_landuse_{distance}m.csv"
    summary.to_csv(out_csv, index=False)
    print(f"Saved: {out_csv}")


In [ ]:
encroachment_classes = [
    "Buildings and other infrastructure",
    "Agriculture",
    "Bauxite extraction / quarry",
    "Plantation",
]

for distance, df in summaries.items():
    risk_pct = df.loc[df["Classify"].isin(encroachment_classes), "% of surrounding ring"].sum()
    print(f"{distance} m ring encroachment-risk share: {risk_pct:.2f}%")


In [ ]:
for distance, df in summaries.items():
    s = df.copy()
    c = s["Classify"].str.lower().str.strip()

    risk_mask = (
        c.str.contains("building|infrastructure") |
        c.str.contains("field") |
        c.str.contains("plantation") |
        c.str.contains("quarry|bauxite")
    )

    risk_pct = s.loc[risk_mask, "% of surrounding ring"].sum()
    total_pct = s["% of surrounding ring"].sum()

    print(f"\n{distance} m ring")
    print(f"Encroachment-risk share: {risk_pct:.2f}%")
    print(f"Total share (should be ~100%): {total_pct:.6f}%")


In [ ]:
# Define class groups (edit names to exactly match your Classify values)
plantation_classes = [
    "Hardwood Plantation: Mahogany",
    "Hardwood Plantation: Mahoe",
    "Hardwood Plantation: Mixed",
    "Plantation: Tree crops, shrub crops, sugar cane"
]

forest_classes = [
    "Closed broadleaved forest (Primary Forest)",
    "Disturbed broadleaved forest (Secondary Forest)",
    "Secondary Forest",
    "Open dry forest - Tall (Woodland/Savanna)",
    "Open dry forest - Short"
]

mixed_forest_and_agriculture_classes = [
    "Fields and Secondary Forest",
    "Fields or Secondary Forest/Pine Plantation",
]

mixed_agriculture_and_bamboo_classes = [ 
    "Fields and Bamboo",
    "Bamboo and Fields"
]

mixed_bamboo_and_forest_classes = [
    "Bamboo and Secondary Forest"
]


In [ ]:


def forest_ring_risk_safe(land, forest_class_list, distances=(100, 1000), simplify_tol=1):
    # Keep only needed columns
    land = land[["Classify", "geometry"]].dropna(subset=["Classify"]).copy()
    land = land[land.geometry.notna()].copy()

    # Fix invalid only (faster than make_valid on all)
    bad = ~land.is_valid
    if bad.any():
        land.loc[bad, "geometry"] = land.loc[bad, "geometry"].make_valid()

    forest = land[land["Classify"].isin(forest_class_list)].copy()
    if forest.empty:
        raise ValueError("No forest features found for the selected class list.")

    # Dissolve once to single geometry
    forest_union = forest.dissolve().geometry.iloc[0]

    outputs = {}
    for d in distances:
        outer = gpd.GeoSeries([forest_union], crs=land.crs).buffer(d).iloc[0]
        ring_geom = outer.difference(forest_union)

        # Optional simplification to reduce geometry complexity
        if simplify_tol and simplify_tol > 0:
            ring_geom = ring_geom.simplify(simplify_tol, preserve_topology=True)

        ring = gpd.GeoDataFrame(geometry=[ring_geom], crs=land.crs)

        # Spatial index prefilter before clip
        idx = list(land.sindex.query(ring_geom, predicate="intersects"))
        cand = land.iloc[idx].copy()

        lu = gpd.clip(cand, ring)
        lu["area_m2"] = lu.geometry.area

        s = lu.groupby("Classify", as_index=False)["area_m2"].sum()
        total_m2 = s["area_m2"].sum()
        s["area_km2"] = s["area_m2"] / 1e6
        s["pct_ring"] = np.where(total_m2 > 0, s["area_m2"] / total_m2 * 100, 0)
        s = s.sort_values("pct_ring", ascending=False)

        c = s["Classify"].str.lower().str.strip()
        risk_mask = (
            c.str.contains("building|infrastructure") |
            c.str.contains("field") |
            c.str.contains("plantation") |
            c.str.contains("quarry|bauxite")
        )
        risk_pct = s.loc[risk_mask, "pct_ring"].sum()

        print(f"\n{d} m ring")
        print(f"Risk share: {risk_pct:.2f}%")
        print(f"Total pct check: {s['pct_ring'].sum():.8f}%")

        outputs[d] = s
        display(s)

        # Free memory each loop
        del cand, lu, s
        gc.collect()

    return outputs


In [ ]:
# # 2) Run smallest test first (100m), with stronger simplification and no big display
# strict_forest = forest_classes

# summaries_strict = forest_ring_risk_safe(
#     terrestrial_landcover,
#     strict_forest,
#     distances=(100,),
#     simplify_tol=5   # safer than 1 for memory
# )


In [ ]:


target_class = "Closed broadleaved forest (Primary Forest)"

# Clean land layer once
land = terrestrial_landcover[["Classify", "geometry"]].dropna(subset=["Classify"]).copy()
land = land[land.geometry.notna()].copy()

bad = ~land.is_valid
if bad.any():
    land.loc[bad, "geometry"] = land.loc[bad, "geometry"].make_valid()

# Select only target forest type
target = land[land["Classify"].str.strip().eq(target_class)].copy()
if target.empty:
    raise ValueError(f"No features found for: {target_class}")

# Dissolve target only (much lighter than all forest classes)
target_union = target.geometry.union_all()

for d in [100]:
    ring_geom = target_union.buffer(d).difference(target_union)
    ring_geom = ring_geom.simplify(5, preserve_topology=True)  # safer memory-wise
    ring = gpd.GeoDataFrame(geometry=[ring_geom], crs=land.crs)

    # Prefilter with spatial index
    idx = list(land.sindex.query(ring_geom, predicate="intersects"))
    cand = land.iloc[idx][["Classify", "geometry"]].copy()

    lu = gpd.clip(cand, ring)
    lu["area_m2"] = lu.geometry.area

    s = lu.groupby("Classify", as_index=False)["area_m2"].sum()
    total_m2 = s["area_m2"].sum()
    s["area_km2"] = s["area_m2"] / 1e6
    s["pct_ring"] = np.where(total_m2 > 0, s["area_m2"] / total_m2 * 100, 0)
    s = s.sort_values("pct_ring", ascending=False)

    c = s["Classify"].str.lower().str.strip()
    risk_mask = (
        c.str.contains("building|infrastructure") |
        c.str.contains("field") |
        c.str.contains("plantation") |
        c.str.contains("quarry|bauxite")
    )
    risk_pct = s.loc[risk_mask, "pct_ring"].sum()

    print(f"\n{target_class} | {d} m ring")
    print(f"Risk share: {risk_pct:.2f}%")
    print(f"Total pct check: {s['pct_ring'].sum():.8f}%")

    display(s.head(20))  # lighter than displaying full table

    out_csv = base_path / f"dphil_paper_3/processed_data/encroachment_{d}m_primary_forest.csv"
    s.to_csv(out_csv, index=False)
    print(f"Saved: {out_csv}")

    del cand, lu, s
    gc.collect()


In [ ]:
def run_one_forest_class(target_class):
    import numpy as np, gc

    land = terrestrial_landcover[["Classify", "geometry"]].dropna(subset=["Classify"]).copy()
    land = land[land.geometry.notna()].copy()

    bad = ~land.is_valid
    if bad.any():
        land.loc[bad, "geometry"] = land.loc[bad, "geometry"].make_valid()

    target = land[land["Classify"].str.strip().eq(target_class)].copy()
    if target.empty:
        raise ValueError(f"No features found for: {target_class}")

    target_union = target.geometry.union_all()

    for d in [100]:
        ring_geom = target_union.buffer(d).difference(target_union).simplify(5, preserve_topology=True)
        ring = gpd.GeoDataFrame(geometry=[ring_geom], crs=land.crs)

        idx = list(land.sindex.query(ring_geom, predicate="intersects"))
        cand = land.iloc[idx][["Classify", "geometry"]].copy()

        lu = gpd.clip(cand, ring)
        lu["area_m2"] = lu.geometry.area

        s = lu.groupby("Classify", as_index=False)["area_m2"].sum()
        total_m2 = s["area_m2"].sum()
        s["area_km2"] = s["area_m2"] / 1e6
        s["pct_ring"] = np.where(total_m2 > 0, s["area_m2"] / total_m2 * 100, 0)
        s = s.sort_values("pct_ring", ascending=False)

        c = s["Classify"].str.lower().str.strip()
        risk_mask = (
            c.str.contains("building|infrastructure") |
            c.str.contains("field") |
            c.str.contains("plantation") |
            c.str.contains("quarry|bauxite")
        )
        risk_pct = s.loc[risk_mask, "pct_ring"].sum()

        print(f"\n{target_class} | {d} m ring")
        print(f"Risk share: {risk_pct:.2f}%")
        print(f"Total pct check: {s['pct_ring'].sum():.8f}%")
        display(s.head(20))

        out_csv = base_path / f"dphil_paper_3/processed_data/encroachment_{d}m_{target_class.replace('/', '_').replace(' ', '_')}.csv"
        s.to_csv(out_csv, index=False)

        del cand, lu, s
        gc.collect()


In [ ]:
run_one_forest_class("Open dry forest - Tall (Woodland/Savanna)")


In [ ]:
run_one_forest_class("Open dry forest - Short")

In [ ]:
run_one_forest_class("Disturbed broadleaved forest (Secondary Forest)")

In [ ]:
import numpy as np
import gc
import geopandas as gpd
from shapely import union_all, set_precision
from shapely.errors import GEOSException

def run_one_forest_class_lowmem(target_class, distance=100, grid_size=20, simplify_tol=20):
    # Keep minimal columns
    land = terrestrial_landcover[["Classify", "geometry"]].dropna(subset=["Classify"]).copy()
    land = land[land.geometry.notna()].copy()

    # Clean land geometries (light touch)
    bad_land = ~land.is_valid
    if bad_land.any():
        land.loc[bad_land, "geometry"] = land.loc[bad_land, "geometry"].buffer(0)

    # Select target class
    target = land[land["Classify"].str.strip().eq(target_class)].copy()
    if target.empty:
        raise ValueError(f"No features found for: {target_class}")

    # Robust target cleaning
    target["geometry"] = target.geometry.make_valid()
    target = target.explode(index_parts=False, ignore_index=True)
    target = target[target.geom_type.isin(["Polygon", "MultiPolygon"])].copy()
    target["geometry"] = target.geometry.buffer(0)
    target = target[target.geometry.notna() & ~target.geometry.is_empty].copy()

    # Safe precision snapping (fallback on failure)
    def safe_set_precision(g, gs):
        try:
            return set_precision(g, gs, mode="pointwise")
        except GEOSException:
            return g

    target["geometry"] = target.geometry.map(lambda g: safe_set_precision(g, grid_size))
    target["geometry"] = target.geometry.buffer(0)
    target = target[target.is_valid & ~target.geometry.is_empty].copy()

    # Simplify to reduce complexity
    target["geometry"] = target.geometry.simplify(simplify_tol, preserve_topology=True)
    target = target[target.geometry.notna() & ~target.geometry.is_empty].copy()

    # Union
    target_union = union_all(target.geometry.values, grid_size=grid_size)

    # Build ring
    outer = target_union.buffer(distance, quad_segs=2)
    ring_geom = outer.difference(target_union).simplify(simplify_tol, preserve_topology=True)
    ring = gpd.GeoDataFrame(geometry=[ring_geom], crs=land.crs)

    # Prefilter by bbox + spatial index
    minx, miny, maxx, maxy = ring_geom.bounds
    cand = land.cx[minx:maxx, miny:maxy].copy()

    if cand.empty:
        raise ValueError("No candidate landcover features intersect ring bounds.")

    idx = list(cand.sindex.query(ring_geom, predicate="intersects"))
    cand = cand.iloc[idx].copy()

    if cand.empty:
        raise ValueError("No candidate landcover features intersect ring geometry.")

    # Clip and summarize
    lu = gpd.clip(cand[["Classify", "geometry"]], ring)
    lu["area_m2"] = lu.geometry.area

    s = lu.groupby("Classify", as_index=False)["area_m2"].sum()
    total_m2 = s["area_m2"].sum()

    s["area_km2"] = s["area_m2"] / 1e6
    s["pct_ring"] = np.where(total_m2 > 0, s["area_m2"] / total_m2 * 100, 0)
    s = s.sort_values("pct_ring", ascending=False)

    # Risk share
    c = s["Classify"].str.lower().str.strip()
    risk_mask = (
        c.str.contains("building|infrastructure") |
        c.str.contains("field") |
        c.str.contains("plantation") |
        c.str.contains("quarry|bauxite")
    )
    risk_pct = s.loc[risk_mask, "pct_ring"].sum()

    print(f"\n{target_class} | {distance} m ring")
    print(f"Risk share: {risk_pct:.2f}%")
    print(f"Total pct check: {s['pct_ring'].sum():.8f}%")

    # Save immediately
    safe_name = (
        target_class.replace("/", "_")
        .replace(" ", "_")
        .replace("(", "")
        .replace(")", "")
    )
    out_csv = base_path / f"dphil_paper_3/processed_data/encroachment_{distance}m_{safe_name}.csv"
    s.to_csv(out_csv, index=False)
    print(f"Saved: {out_csv}")

    # Cleanup
    del land, target, cand, lu
    gc.collect()

    return s

# Run for Secondary Forest (100 m only)
s_secondary = run_one_forest_class_lowmem("Secondary Forest", distance=100, grid_size=20, simplify_tol=20)
display(s_secondary.head(20))


In [ ]:

processed_dir = base_path / "dphil_paper_3/processed_data"
files = sorted(processed_dir.glob("encroachment_*m_*.csv"))

pat = re.compile(r"encroachment_(\d+)m_(.+)\.csv$")

long_parts = []
summary_rows = []

for f in files:
    m = pat.match(f.name)
    if not m:
        continue

    distance_m = int(m.group(1))
    forest_key = m.group(2)  # safe filename version of class

    df = pd.read_csv(f)
    if "pct_ring" not in df.columns:
        continue

    df["distance_m"] = distance_m
    df["forest_key"] = forest_key
    long_parts.append(df)

    c = df["Classify"].astype(str).str.lower().str.strip()
    risk_mask = (
        c.str.contains("building|infrastructure") |
        c.str.contains("field") |
        c.str.contains("plantation") |
        c.str.contains("quarry|bauxite")
    )

    summary_rows.append({
        "forest_key": forest_key,
        "distance_m": distance_m,
        "risk_share_pct": df.loc[risk_mask, "pct_ring"].sum(),
        "total_pct_check": df["pct_ring"].sum(),  # should be ~100
        "top_neighbor_class": df.sort_values("pct_ring", ascending=False).iloc[0]["Classify"],
        "top_neighbor_pct": df["pct_ring"].max(),
    })

all_encroachment_long = pd.concat(long_parts, ignore_index=True) if long_parts else pd.DataFrame()
encroachment_summary = pd.DataFrame(summary_rows).sort_values(["forest_key", "distance_m"])

# Optional: make forest name prettier
encroachment_summary["forest_class"] = (
    encroachment_summary["forest_key"]
    .str.replace("_", " ", regex=False)
)

display(encroachment_summary)

# Optional wide table (risk % by distance)
risk_wide = encroachment_summary.pivot(index="forest_class", columns="distance_m", values="risk_share_pct")
display(risk_wide)

# Save combined outputs
all_encroachment_long.to_csv(processed_dir / "encroachment_all_results_long.csv", index=False)
encroachment_summary.to_csv(processed_dir / "encroachment_summary.csv", index=False)


In [ ]:

df = all_encroachment_long.copy()

def norm(s):
    s = str(s).strip().lower()
    s = re.sub(r"\s+", " ", s)
    return s

agriculture = {
    norm("Fields: Herbaceous crops, fallow, cultivated vegetables"),
    norm("Fields: Pasture,Human disturbed, grassland"),
    norm("Fields: Bare Land"),
}

plantations = {
    norm("Plantation: Tree crops, shrub crops, sugar cane, banana"),
    norm("Hardwood Plantation: Mahogany"),
    norm("Hardwood Plantation: Euculytus"),
    norm("Hardwood Plantation: Mahoe"),
    norm("Hardwood Plantation: Mixed"),
}

mixed_threat = {
    norm("Bamboo and Fields"),
    norm("Bamboo and Secondary Forest"),
    norm("Fields and Bamboo"),
    norm("Fields and Secondary Forest"),
    norm("Fields or Secondary Forest/Pine Plantation"),
}

development = {norm("Buildings and other infrastructures")}

forest = {
    norm("Closed broadleaved forest (Primary Forest)"),
    norm("Disturbed broadleaved forest (Secondary Forest)"),
    norm("Open dry forest - Short"),
    norm("Open dry forest - Tall (Woodland/Savanna)"),
    norm("Secondary Forest"),
}

other_natural = {
    norm("Mangrove Forest"),
    norm("Swamp Forest"),
    norm("Water Body"),
    norm("Herbaceous Wetland"),
}

bamboo = {norm("Bamboo")}
mining = {norm("Bauxite Extraction"), norm("Bauxite extraction / quarry"), norm("Quarry")}
bare_rock = {norm("Bare Rock")}

def classify_category(classify_value):
    c = norm(classify_value)
    if c in development: return "development"
    if c in agriculture: return "agriculture"
    if c in plantations: return "plantations"
    if c in mixed_threat: return "mixed_threat"
    if c in forest: return "forest"
    if c in other_natural: return "other_natural"
    if c in bamboo: return "bamboo"
    if c in mining: return "mining"
    if c in bare_rock: return "bare_rock"
    return "other_unmapped"

df["category"] = df["Classify"].apply(classify_category)

print("Unmapped classes:")
print(sorted(df.loc[df["category"] == "other_unmapped", "Classify"].dropna().unique()))

cat = (
    df.groupby(["forest_key", "distance_m", "category"], as_index=False)["area_m2"]
      .sum()
)
cat["pct_of_ring"] = (
    cat["area_m2"] /
    cat.groupby(["forest_key", "distance_m"])["area_m2"].transform("sum") * 100
)

cat_100m = cat[cat["distance_m"] == 100].copy()
display(cat_100m.sort_values(["forest_key", "pct_of_ring"], ascending=[True, False]))

cat_100m_wide = cat_100m.pivot_table(
    index="forest_key",
    columns="category",
    values="pct_of_ring",
    fill_value=0
)
display(cat_100m_wide)

cat.to_csv(base_path / "dphil_paper_3/processed_data/encroachment_category_summary.csv", index=False)
cat_100m_wide.to_csv(base_path / "dphil_paper_3/processed_data/encroachment_category_summary_100m_wide.csv")
